#Task 1: Conceptual Understanding

1. What is the difference between "Love" and "love" in NLP?
In raw text, "Love" and "love" are treated as two completely distinct tokens by a computer because they have different ASCII/Unicode values due to the capitalization. If not standardized (lowercased), a machine learning model will unnecessarily split its learned weights between the two, diluting the perceived importance of the word.

2. What happens if stopwords are not removed?
If common stopwords (like "the", "is", "in") are kept, they will dominate the frequency distributions of your dataset. This increases the dimensionality of your data, consumes more memory and compute power, and can add "noise," causing models to focus on structurally necessary but semantically meaningless words rather than the core context.

3. Provide two real-world scenarios where removing stopwords can be harmful.

Sentiment Analysis: Removing words like "not" or "no" can completely invert the meaning of a sentence (e.g., "The movie was not good" becomes "movie good").

Machine Translation & Text Generation: Generative models and translators need grammatical glue to produce human-readable, syntactically correct sentences. Stripping stopwords degrades the fluency of the output.

4. Explain the difference between stemming and lemmatization.

Stemming is a crude, rule-based heuristic that chops off the suffixes of words to find their root (e.g., "caring" becomes "car"). It is fast but often results in non-existent words.

Lemmatization is a sophisticated process that uses vocabulary and morphological analysis to accurately reduce a word to its dictionary base form, known as the lemma (e.g., "better" becomes "good", "caring" becomes "care")

In [ ]:
import re
from collections import Counter
import pandas as pd

# ==========================================
# Task 2: Build Advanced Preprocessing Function
# ==========================================
def preprocess_text(text):
    """
    Cleans raw text by removing numbers, URLs, extra spaces, and repeated characters,
    while converting to lowercase and filtering short tokens.
    """
    # Task 7: Error Handling - Handle empty strings or non-string inputs
    if not isinstance(text, str) or not text.strip():
        return [], ""

    # 6. Remove URLs and email-like patterns
    text = re.sub(r'http[s]?://\S+|www\.\S+|\S+@\S+', '', text)

    # 1. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 3. Handle repeated characters (reduce 3 or more consecutive identical characters to 1)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove punctuation and emojis (keeps only alphanumeric and whitespace)
    # This inherently handles the "Only emojis" error handling requirement.
    text = re.sub(r'[^\w\s]', '', text)

    # 4. Convert text to lowercase
    text = text.lower()

    # 2. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    raw_tokens = text.split()

    # 5. Remove very short tokens (length <= 2), keeping meaningful exceptions
    exceptions = {"no", "not", "ok"}
    tokens = [word for word in raw_tokens if len(word) > 2 or word in exceptions]

    # Reconstruct cleaned sentence
    clean_sentence = " ".join(tokens)

    return tokens, clean_sentence

# ==========================================
# Task 6: Build Full Pipeline
# ==========================================
def full_pipeline(text_list):
    """
    Processes a list of text strings and returns a dictionary of tokens and clean sentences.
    """
    all_tokens = []
    all_clean_sentences = []

    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        # Only append if not empty (handles empty/only-number/only-emoji inputs)
        if tokens:
            all_tokens.extend(tokens)
        all_clean_sentences.append(clean_sentence)

    return {
        "tokens": all_tokens,
        "clean_sentences": all_clean_sentences
    }

# ==========================================
# Task 3: Stress Testing
# ==========================================
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
    "",               # Task 7: Empty string
    "😀😃😄😁",        # Task 7: Only emojis
    "1234567890"      # Task 7: Only numbers
]

print("--- TASK 3: STRESS TESTING ---")
results = []
for original in sample_inputs:
    tokens, cleaned = preprocess_text(original)
    results.append({
        "Original Text": original,
        "Cleaned Tokens": tokens,
        "Cleaned Sentence": cleaned
    })

    # Print formatted output
    print(f"Original: {original}")
    print(f"Tokens:   {tokens}")
    print(f"Cleaned:  {cleaned}\n")

# ==========================================
# Task 4: Token Analytics
# ==========================================
print("--- TASK 4: TOKEN ANALYTICS ---")
analytics_data = []

for item in results:
    tokens = item["Cleaned Tokens"]
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length = sum(len(word) for word in tokens) / total_tokens if total_tokens > 0 else 0

    analytics_data.append({
        "Text": item["Original Text"][:30] + "...", # Truncated for display
        "Total": total_tokens,
        "Unique": unique_tokens,
        "Avg Length": round(avg_length, 2)
    })

# Display as DataFrame for clean formatting
df_analytics = pd.DataFrame(analytics_data)
print(df_analytics.to_string(index=False))

# ==========================================
# Task 5: Frequency Analysis
# ==========================================
print("--- TASK 5: FREQUENCY ANALYSIS ---")
pipeline_results = full_pipeline(sample_inputs)
all_combined_tokens = pipeline_results["tokens"]

word_counts = Counter(all_combined_tokens)

print("Top 10 most frequent words:")
for word, count in word_counts.most_common(10):
    print(f" - {word}: {count}")

print("\nTop 5 least frequent words:")
# Reversing the most_common list to get the least frequent
least_frequent = word_counts.most_common()[:-6:-1]
for word, count in least_frequent:
    print(f" - {word}: {count}")

--- TASK 3: STRESS TESTING ---
Original: Get 100% FREE access now!!!
Tokens:   ['get', 'free', 'access', 'now']
Cleaned:  get free access now

Original: I absolutely looooved this product
Tokens:   ['absolutely', 'loved', 'this', 'product']
Cleaned:  absolutely loved this product

Original: Worst service ever... 0/10
Tokens:   ['worst', 'service', 'ever']
Cleaned:  worst service ever

Original: Call me at 9876543210
Tokens:   ['call']
Cleaned:  call

Original: This is THE best course!!!
Tokens:   ['this', 'the', 'best', 'course']
Cleaned:  this the best course

Original: Visit https://openai.com now!
Tokens:   ['visit', 'now']
Cleaned:  visit now

Original: Nooooo this is baaad!!!
Tokens:   ['no', 'this', 'bad']
Cleaned:  no this bad

Original: OK OK OK I got it
Tokens:   ['ok', 'ok', 'ok', 'got']
Cleaned:  ok ok ok got

Original: Win $$$ now!!! Limited offer!!!
Tokens:   ['win', 'now', 'limited', 'offer']
Cleaned:  win now limited offer

Original: I am not happy with this
Tokens:   ['